In [ ]:
from openai import OpenAI


client = OpenAI(
    api_key="o0t54t4a1asga4k1000dl7zxe8f3k1s2p02cvz27",
    base_url="https://api.gpugeek.com/v1"
)


response = client.chat.completions.create(
    model="DeepSeek-V4-Flash",
    messages=[
        {
            "role":"user",
            "content":
            "你好"
        }
    ]
)


print(
    response.choices[0].message.content
)

Transformer模型的优化方法非常丰富，涵盖了从**训练阶段**到**推理阶段**，从**算法层面**到**工程/硬件层面**的多个维度。

下面我将从几个核心方向为你系统梳理这些优化方法，并区分其适用场景（训练优化 vs. 推理优化）。

### 一、训练优化

主要目标是**加速收敛、降低显存占用、提高训练稳定性**。

#### 1. 学习率调度与优化器
-   **Noam Scheduler（预热 + 衰减）**：这是Transformer的标配。先线性增加学习率（预热阶段，防止模型参数剧烈震荡），再按步长平方根倒数衰减。这是训练稳定性的基石。
-   **AdamW**：比标准Adam效果更好，因为它将权重衰减与学习率解耦，能有效防止过拟合。
-   **混合精度训练（AMP）**：使用FP16进行前向和反向传播，FP32维护主权重副本。利用NVIDIA GPU的Tensor Core，可将训练速度提升2-3倍，同时显存占用减半。**几乎已成为现代训练的标准配置**。
-   **梯度累积**：当模型很大，单卡无法容纳足够大的batch size时，通过累积多个小batch的梯度，模拟大batch的效果，从而稳定训练。

#### 2. 注意力机制优化（核心）
-   **FlashAttention**：**这是近年来最重要的优化之一**。它通过分块（tiling）和重计算（recomputation）技术，避免了将完整的$N \times N$注意力矩阵写入到高带宽显存（HBM），而是利用计算速度更快的SRAM。这使得训练速度显著提升，并且显存占用从$O(N^2)$降低到近乎线性。
-   **稀疏注意力**：对于长序列，不计算所有token对之间的注意力，而是只计算局部（如窗口注意力）、全局（如特定token）或哈希分组后的相似块。代表方法是**BigBird**、**Longformer**、**Reformer**。
-   **线性注意力**：用核方法（如$\phi(Q)\phi(K)^T$）近似Softmax，将复杂度从$O(N^2)$降为$O(N)$。代表方法：**Performer**、**Linear Transformer**。这类方法在极长序列上优势明显，但精度可能略有损失。

#### 3. 模型架构与参数优化
-   *